###Notebook-0: Data-acquisition of "Die Bombe"
For the acquisition of the relevant data - in this case, each page of all the issues of the viennese periodical "Die Bombe" - the provided IIIF-manifests of the austrian national library will be used. The CSV-file provides the manifests to all issues of the periodical, which is published via the [ANNO catalogue](https://anno.onb.ac.at/) of the national library.

This Notebook will download each Canvas/page of the periodical in jpg-format and create a metadata-sheet with the following information:

- Page-ID: unique identifier of each single page (Anno-ID+00+number of page)
- Anno-ID: ID assigned per issue by the ONB
- Anno-URL: linking to the issue in the ANNO-catalogue
- Periodical-title: IIIF-manifest: metadata --> label/en/"title" --> value/en
- Date: IIIF-manifest: metadata --> label/en/"year/date" --> value/en
- Publishing-Place: IIIF-manifest: metadata --> label/en/"place" --> value/en
- Language: IIIF-manifest: metadata --> label/en/"languages" --> value/en
- Mediatype: IIIF-manifest: metadata --> label/en/"mediatype" --> value/en
- Keywords: IIIF-manifest: metadata --> label/en/"keywords" --> value/en
- Attribution: IIIF-manifest: requiredStatement --> label/en/"Attribution" --> value/en
- Provider: IIIF-manifest: requiredStatement --> label/en/"Provider" --> value/en
- Rights: IIIF-manifest: "Rights"

##Environment
This notebook was created with the help of ChatGPT-5.5 and is supposed to be used in a Google Colab/Google Drive environment.

## 1) Installations, Imports and Path Configurations
 - Installation of needed libraries
 - Imports from libraries
 - Path-settings
 - Google Drive mount

In [ ]:
# === Installations ===
!pip -q install pandas requests tqdm

# === Imports & Setup ===
import time, json, re
from pathlib import Path
import pandas as pd
import requests
from tqdm import tqdm
from PIL import Image
import shutil
from datetime import datetime

# ========== DRIVE / Path-Config ==========
USE_GOOGLE_DRIVE = True
DATA_OUTPUT_DIR = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE"
CSV_PATH = "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/data_gollner.csv"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

#Creation of the base-directory and the subordinated metadata- and log-directory
OUT_BASE   = Path(DATA_OUTPUT_DIR)
OUT_PAGES  = OUT_BASE / "pages_jpg"     # JPGs
OUT_META   = OUT_BASE / "metadata"      # CSV
OUT_LOGS   = OUT_BASE / "logs"
for p in (OUT_PAGES, OUT_META, OUT_LOGS):
    p.mkdir(parents=True, exist_ok=True)

STATE_FILE = OUT_LOGS / "state.json"  # creates a log for a next_start, if the notebook cannot download eveything at once
MASTER_LOG = OUT_LOGS / "download_log_master.csv"
PAGES_CSV  = OUT_META / "pages-jpg_metadata.csv"

# ========== BATCH-Config ==========
BATCH_SIZE = 200
RESET_STATE = False
MANUAL_START_IDX = None
REWIND_BATCHES = 0

# ========== TESTING-Config ==========
MAX_TOTAL_PAGES = None   # set to None for full run

# ========== DOWNLOAD-Config ==========
SIZE_PARAM = "max"     # full resolution of available jpgs
MAX_SLEEP = 0.1        # break to let the system rest

# ========== CSV-columns (change if needed) ==========
MANIFEST_COL = "iiif_manifest"
ANNO_ID_COL  = "anno_id"
ANNO_URL_COL = "anno_url"

# ========== FIXED METADATA ==========
# column values for metadata-file which should remain throughout:
FIXED_PERIODICAL_TITLE = "Die Bombe"

# ========== HTTP-Session ==========
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Colab-IIIF-JPG-Downloader/2.4"})


Mounted at /content/drive


Output of the cell above will be the following directory structure:
1) pages_jpg: containing the jpg-files of each page in a sub-directory of each issue
2) metadata: containing the metadata-file (where one row describes one page)
3) logs: where log-files are stored for resumable downloading

## 2) Auxiliary functions

Functions used for extracting relevant metadata from the manifests and for downloading the jpg-files.

In [ ]:
def get_json(url, tries=3, timeout=60):
    last = None
    for i in range(tries):
        try:
            r = SESSION.get(url, timeout=timeout)
            if r.ok:
                return r.json()
            last = f"{r.status_code} {r.text[:200]}"
        except Exception as e:
            last = str(e)
        time.sleep(0.6 * (i + 1))
    raise RuntimeError(f"GET JSON failed: {url} :: {last}")

def pick_lang_value(lang_map, preferred=("en", "none", "de")):
    if not isinstance(lang_map, dict):
        return None
    for lang in preferred:
        arr = lang_map.get(lang)
        if isinstance(arr, list) and arr:
            return arr[0]
    for _, arr in lang_map.items():
        if isinstance(arr, list) and arr:
            return arr[0]
    return None

def get_manifest_metadata_en(manifest: dict, wanted_label_en: str):
    wanted = wanted_label_en.strip().lower()
    for md in manifest.get("metadata", []) or []:
        lab_en = pick_lang_value(md.get("label") or {}, preferred=("en",))
        if (lab_en or "").strip().lower() == wanted:
            return pick_lang_value(md.get("value") or {}, preferred=("en", "none", "de"))
    return None

def get_required_statement_value_en(manifest: dict, wanted_label_en: str):
    rs = manifest.get("requiredStatement")
    if not isinstance(rs, dict):
        return None
    lab_en = pick_lang_value(rs.get("label") or {}, preferred=("en",))
    if (lab_en or "").strip().lower() == wanted_label_en.strip().lower():
        return pick_lang_value(rs.get("value") or {}, preferred=("en", "none", "de"))
    return None

def get_provider_label_en(manifest: dict):
    prov = (manifest.get("provider") or [])
    if not prov:
        return None
    return pick_lang_value((prov[0].get("label") or {}), preferred=("en", "none", "de"))

def iter_canvases(m: dict):
    if "items" in m:
        for c in m["items"]:
            if isinstance(c, dict) and c.get("type") == "Canvas":
                yield c
    for seq in m.get("sequences", []) or []:
        for c in seq.get("canvases", []) or []:
            yield c

def extract_image_service_and_body_url(canvas: dict):
    if "items" in canvas:
        try:
            body = canvas["items"][0]["items"][0]["body"]
            svc = body.get("service")
            if isinstance(svc, list) and svc:
                svc_id = svc[0].get("id") or svc[0].get("@id")
            elif isinstance(svc, dict):
                svc_id = svc.get("id") or svc.get("@id")
            else:
                svc_id = None
            return svc_id, (body.get("id") or body.get("@id"))
        except Exception:
            pass
    if "images" in canvas:
        try:
            res = canvas["images"][0]["resource"]
            svc = res.get("service") or {}
            return (svc.get("@id") or svc.get("id")), (res.get("@id") or res.get("id"))
        except Exception:
            pass
    return None, None

def build_jpg_url(service_id: str, size_param: str) -> str:
    return f"{service_id}/full/{size_param}/0/default.jpg"

def download_to(out_path, url, timeout=30):
    """
    Safe download:
    - Downloads to temporary *.tmp file
    - Only renames to final file if complete
    - Prevents truncated JPGs after crashes
    """
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")

    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            expected_size = int(r.headers.get("Content-Length", 0))

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        # Verify size if Content-Length available
        actual_size = tmp_path.stat().st_size
        if expected_size and actual_size != expected_size:
            tmp_path.unlink(missing_ok=True)
            return False

        # Replace existing file safely
        tmp_path.replace(out_path)
        return True

    except Exception:
        tmp_path.unlink(missing_ok=True)
        return False

def append_csv(path, rows, columns=None, encoding="utf-8"):
    """
    Append rows (list[dict]) to a CSV in a stable, reproducible column order.

    - If `columns` is provided: enforce exactly that order (missing cols become empty).
    - If `columns` is None:
        - If file exists: reuse its existing header order.
        - If file doesn't exist: infer order from first row's key order.
    - Automatically adds any "extra" keys (not in columns/header) at the end.
    """
    path = Path(path)
    if not rows:
        return

    df = pd.DataFrame(rows)

    # Determine target column order
    if columns is not None:
        col_order = list(columns)
    else:
        if path.exists() and path.stat().st_size > 0:
            # Read header only, preserve existing column order
            existing = pd.read_csv(path, nrows=0, encoding=encoding)
            col_order = list(existing.columns)
        else:
            # New file: respect insertion order of keys in the first row
            col_order = list(rows[0].keys())

    # Append any new/unseen columns at the end (so nothing gets dropped)
    for c in df.columns:
        if c not in col_order:
            col_order.append(c)

    # Reindex for stable order; missing columns become NaN/empty
    df = df.reindex(columns=col_order)

    # Write (append) with correct header handling
    write_header = not (path.exists() and path.stat().st_size > 0)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, mode="a", index=False, header=write_header, encoding=encoding)

def save_state(next_start: int):
    STATE_FILE.write_text(json.dumps({"next_start": int(next_start)}))

def parse_place_and_gnd(place_html: str | None):
    """
    Input example (already unescaped by JSON):
      <span>Wien -- (GND: <a href="http://d-nb.info/gnd/4066009-6">4066009-6</a>)</span>
    Output:
      ("Wien", "http://d-nb.info/gnd/4066009-6")  # or (place, None) if no GND link
    """
    if not place_html:
        return None, None

    txt = str(place_html)

    # Place: take text right after <span> up to " --" (or closing tag as fallback)
    m_place = re.search(r"<span>\s*([^<]+?)(?:\s*--|\s*</span>)", txt, flags=re.IGNORECASE)
    place = m_place.group(1).strip() if m_place else None

    # GND link: take href
    m_href = re.search(r'href="([^"]*d-nb\.info/gnd/[^"]+)"', txt, flags=re.IGNORECASE)
    gnd_link = m_href.group(1).strip() if m_href else None

    return place, gnd_link

## 3) Loading the CSV and starting the download

In [ ]:
# ========== loading provided CSV ==========
df = pd.read_csv(CSV_PATH)
for col in (MANIFEST_COL, ANNO_ID_COL, ANNO_URL_COL):
    if col not in df.columns:
        raise ValueError(f"Column '{col}' missing in CSV. Available: {list(df.columns)}")

N = len(df)
print(f"Datasets (issues): {N} | Manifest: {MANIFEST_COL} | Anno-ID: {ANNO_ID_COL} | Anno-URL: {ANNO_URL_COL}")

# ========== Stable column order for the PAGES metadata CSV ==========
PAGES_COLUMNS = [
    # page-level (first)
    "page_id",
    "page_number",
    "iiif_jpg_url",
    "local_jpg_path",
    "width",
    "height",
    "download_status",
    # issue-level (repeated per page)
    "anno_id",
    "anno_url",
    "manifest_url",
    "periodical_title",
    "date",
    "publishing_place",
    "publishing_place_gnd",
    "language",
    "mediatype",
    "attribution",
    "provider",
    "rights",
]

# ========== Determine starting position (state + knobs) ==========
start_idx = 0
if STATE_FILE.exists():
    try:
        start_idx = json.loads(STATE_FILE.read_text()).get("next_start", 0)
    except Exception:
        start_idx = 0

# Apply configurations from Batch-config
if RESET_STATE:
    start_idx = 0
if MANUAL_START_IDX is not None:
    start_idx = max(0, min(N, int(MANUAL_START_IDX)))
if REWIND_BATCHES > 0:
    start_idx = max(0, start_idx - REWIND_BATCHES * BATCH_SIZE)

# Persist the chosen start_idx so the loop below continues from there
save_state(start_idx)

print(f"Starting at row index: {start_idx} (BATCH_SIZE={BATCH_SIZE})")

# ========== Running all Batches ==========
# Writes metadata + logs + state after each issue and metadata after each page.
# If Colab disconnects, re-running this cell resumes from STATE_FILE['next_start'].

total_pages_downloaded = 0  # for MAX_TOTAL_PAGES testing limit (per run)

while True:
    # Load current position from state (authoritative for resume)
    try:
        cur = json.loads(STATE_FILE.read_text()).get("next_start", 0) if STATE_FILE.exists() else 0
    except Exception:
        cur = 0

    if cur >= N:
        print("\n✅ All issues processed (cur >= N).")
        break

    end_idx = min(cur + BATCH_SIZE, N)
    print(f"\n========== BATCH ==========")
    print(f"Processing issues rows {cur}..{end_idx-1} of {N-1} (BATCH_SIZE={BATCH_SIZE})")

    df_batch = df.iloc[cur:end_idx].copy()

    for row_pos_in_batch, (row_idx, row) in enumerate(
        tqdm(df_batch.iterrows(), total=len(df_batch), desc="Output (Batch)"),
        start=0
    ):
        # IMPORTANT: resume should be position-based, not index-based
        global_pos = cur + row_pos_in_batch  # absolute positional index in full df

        manifest_url = row.get(MANIFEST_COL, "")
        anno_id = str(row.get(ANNO_ID_COL, "")).strip()
        anno_url = str(row.get(ANNO_URL_COL, "")).strip()

        issue_log = {"row": int(global_pos), "anno_id": anno_id}

        if pd.isna(manifest_url) or not str(manifest_url).strip():
            issue_log.update({"pages_downloaded": 0, "status": "skip-empty-manifest"})
            append_csv(MASTER_LOG, [issue_log])
            save_state(global_pos + 1)
            continue

        manifest_url = str(manifest_url).strip()

        # loading the manifest
        try:
            m = get_json(manifest_url)
        except Exception as e:
            issue_log.update({"pages_downloaded": 0, "status": f"manifest-fail: {e}"})
            append_csv(MASTER_LOG, [issue_log])
            save_state(global_pos + 1)
            continue

        # Place parsing: keep only "Wien" and GND link in separate columns
        place_raw = get_manifest_metadata_en(m, "Place")
        publishing_place, publishing_place_gnd = parse_place_and_gnd(place_raw)

        # Issue-level metadata (repeated per page)
        issue_meta = {
            "anno_id": anno_id,
            "anno_url": anno_url,
            "manifest_url": manifest_url,
            # Title should not include date: fixed value for this corpus
            "periodical_title": FIXED_PERIODICAL_TITLE,
            "date": get_manifest_metadata_en(m, "Year/Date"),
            "publishing_place": publishing_place,
            "publishing_place_gnd": publishing_place_gnd,
            "language": get_manifest_metadata_en(m, "Languages"),
            "mediatype": get_manifest_metadata_en(m, "Mediatype"),
            "attribution": get_required_statement_value_en(m, "Attribution"),
            "provider": get_provider_label_en(m),
            "rights": m.get("rights"),
        }

        # Downloading the pages and metadata append per download
        pages_downloaded = 0
        issue_dir = OUT_PAGES / anno_id
        issue_dir.mkdir(parents=True, exist_ok=True)

        for page_idx, canv in enumerate(iter_canvases(m), start=1):
            # --- Global page limit for testing ---
            if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
                break

            page_id = f"{anno_id}_{page_idx:03d}"  # e.g. bom18710108_001

            svc_id, body_id = extract_image_service_and_body_url(canv)

            jpg_url = None
            if svc_id:
                jpg_url = build_jpg_url(svc_id, SIZE_PARAM)
            elif body_id and str(body_id).lower().endswith((".jpg", ".jpeg")):
                jpg_url = body_id

            if not jpg_url:
                page_row = {
                    **issue_meta,
                    "page_id": page_id,
                    "page_number": page_idx,
                    "iiif_jpg_url": None,
                    "local_jpg_path": None,
                    "width": canv.get("width"),
                    "height": canv.get("height"),
                    "download_status": "no-image-url",
                }
                append_csv(PAGES_CSV, [page_row], columns=PAGES_COLUMNS)
                continue

            out_path = issue_dir / f"{page_id}.jpg"
            ok = download_to(out_path, jpg_url)

            page_row = {
                **issue_meta,
                "page_id": page_id,
                "page_number": page_idx,
                "iiif_jpg_url": jpg_url,
                "local_jpg_path": str(out_path) if ok else None,
                "width": canv.get("width"),
                "height": canv.get("height"),
                "download_status": "ok" if ok else "download-fail",
            }
            append_csv(PAGES_CSV, [page_row], columns=PAGES_COLUMNS)

            if ok:
                pages_downloaded += 1
                total_pages_downloaded += 1

            if MAX_SLEEP:
                time.sleep(MAX_SLEEP)

        # Issue log row + state written immediately
        issue_log.update({
            "pages_downloaded": pages_downloaded,
            "status": "ok" if pages_downloaded > 0 else "no-pages",
        })
        append_csv(MASTER_LOG, [issue_log])

        # Always advance (so a problematic issue doesn't block progress)
        save_state(global_pos + 1)

        # --- Stop after reaching global page limit ---
        if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
            print(f"\nReached MAX_TOTAL_PAGES = {MAX_TOTAL_PAGES}. Stopping test run.")
            break

    # If we stopped due to MAX_TOTAL_PAGES, stop the outer loop too
    if MAX_TOTAL_PAGES is not None and total_pages_downloaded >= MAX_TOTAL_PAGES:
        break

print("\nFinished download loop.")
print(f"Pages (JPG): {OUT_PAGES}")
print(f"Pages metadata CSV: {PAGES_CSV}")
print(f"Master-Log: {MASTER_LOG}")
print(f"State: {STATE_FILE}")
if MAX_TOTAL_PAGES is not None:
    print(f"Total pages downloaded this run: {total_pages_downloaded} (limit: {MAX_TOTAL_PAGES})")


Datasets (issues): 2593 | Manifest: iiif_manifest | Anno-ID: anno_id | Anno-URL: anno_url
Starting at row index: 2593 (BATCH_SIZE=200)

✅ All issues processed (cur >= N).

Finished download loop.
Pages (JPG): /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/pages_jpg
Pages metadata CSV: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv
Master-Log: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/logs/download_log_master.csv
State: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/logs/state.json


##4) (Optional) Checking the download-count and retrying failed downloads

The cells below will count the number of downloaded pages as well as compare the issue folder names against the anno-IDs with which they coincide. Should issues/pages still be missing, lists with the corresponding IDs will be made to make re-downloading possible. The cells also save metadata-files as backup prior to redownloading and makes an integrity check that all numbers of issues and pages match.

In [ ]:
# =================================================
# Integrity Check: How many pages were downloaded?
# =================================================

# Count actual downloaded image files on disk
img_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
all_imgs = [p for p in OUT_PAGES.rglob("*") if p.is_file() and p.suffix.lower() in img_exts]

issue_dirs = sorted([p for p in OUT_PAGES.iterdir() if p.is_dir()])

print("========== DOWNLOAD INTEGRITY CHECK ==========")
print(f"Pages folder: {OUT_PAGES}")
print(f"Issue folders found: {len(issue_dirs)}")
print(f"Downloaded image files found: {len(all_imgs)}")

# Cross-check with metadata CSV
if PAGES_CSV.exists() and PAGES_CSV.stat().st_size > 0:
    import pandas as pd
    df_pages = pd.read_csv(PAGES_CSV, dtype=str).fillna("")
    total_rows = len(df_pages)
    ok = (df_pages.get("download_status", "") == "ok").sum() if "download_status" in df_pages.columns else None
    print("\n========== METADATA CROSS-CHECK ==========")
    print(f"Rows in pages metadata CSV: {total_rows}")
    if ok is not None:
        print(f"Rows with download_status == 'ok': {ok}")
        print(f"Rows with non-ok status: {total_rows - ok}")
else:
    print("\n(No pages metadata CSV found yet; run the download cell first.)")


========== DOWNLOAD INTEGRITY CHECK ==========
Pages folder: /content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/pages_jpg
Issue folders found: 2593
Downloaded image files found: 21303

========== METADATA CROSS-CHECK ==========
Rows in pages metadata CSV: 21360
Rows with download_status == 'ok': 21314
Rows with non-ok status: 46


In [ ]:
# ============================================================
# FAST LOCAL PAGE COMPLETENESS CHECK
# ============================================================

PAGE_REPAIR_REPORT = OUT_META / "missing_pages_to_redownload.csv"

# Files smaller than this are treated as suspicious/incomplete.
MIN_FILE_SIZE_BYTES = 1024


# ------------------------------------------------------------
# Load the page metadata created during data acquisition
# ------------------------------------------------------------

if not PAGES_CSV.exists():
    raise FileNotFoundError(
        f"Page metadata CSV not found:\n{PAGES_CSV}\n\n"
        "The fast check requires the page metadata CSV."
    )

df_pages = pd.read_csv(PAGES_CSV, dtype=str).fillna("")

required_columns = {
    "anno_id",
    "page_id",
    "page_number",
    "iiif_jpg_url",
}

missing_columns = required_columns - set(df_pages.columns)

if missing_columns:
    raise ValueError(
        "The page metadata CSV is missing these required columns: "
        + ", ".join(sorted(missing_columns))
    )


# ------------------------------------------------------------
# Remove duplicate metadata rows
# ------------------------------------------------------------

df_pages["anno_id"] = df_pages["anno_id"].str.strip()
df_pages["page_id"] = df_pages["page_id"].str.strip()

df_pages = (
    df_pages[
        df_pages["anno_id"].ne("")
        & df_pages["page_id"].ne("")
    ]
    .drop_duplicates(subset=["page_id"], keep="last")
    .reset_index(drop=True)
)

print("Indexing locally downloaded JPG files...")

local_files = {}

for jpg_path in tqdm(
    OUT_PAGES.rglob("*.jpg"),
    desc="Indexing JPG files"
):
    try:
        local_files[jpg_path.stem] = {
            "path": jpg_path,
            "size": jpg_path.stat().st_size,
        }
    except OSError:
        # The file exists but could not be inspected.
        local_files[jpg_path.stem] = {
            "path": jpg_path,
            "size": -1,
        }


# ------------------------------------------------------------
# Compare metadata against local files
# ------------------------------------------------------------

missing_page_rows = []

expected_pages = len(df_pages)
valid_pages = 0
missing_files = 0
small_files = 0
unreadable_files = 0

for row in tqdm(
    df_pages.itertuples(index=False),
    total=len(df_pages),
    desc="Checking expected pages"
):
    row_dict = row._asdict()

    anno_id = str(row_dict.get("anno_id", "")).strip()
    page_id = str(row_dict.get("page_id", "")).strip()

    expected_path = OUT_PAGES / anno_id / f"{page_id}.jpg"

    file_info = local_files.get(page_id)

    if file_info is None:
        reason = "file-missing"
        missing_files += 1

    elif file_info["size"] < 0:
        reason = "file-unreadable"
        unreadable_files += 1

    elif file_info["size"] < MIN_FILE_SIZE_BYTES:
        reason = "file-too-small"
        small_files += 1

    else:
        valid_pages += 1
        continue

    missing_page_rows.append({
        "anno_id": anno_id,
        "anno_url": row_dict.get("anno_url", ""),
        "manifest_url": row_dict.get("manifest_url", ""),
        "page_id": page_id,
        "page_number": row_dict.get("page_number", ""),
        "iiif_jpg_url": row_dict.get("iiif_jpg_url", ""),
        "expected_local_path": str(expected_path),
        "width": row_dict.get("width", ""),
        "height": row_dict.get("height", ""),
        "reason": reason,
    })


# ------------------------------------------------------------
# Create report
# ------------------------------------------------------------

report_columns = [
    "anno_id",
    "anno_url",
    "manifest_url",
    "page_id",
    "page_number",
    "iiif_jpg_url",
    "expected_local_path",
    "width",
    "height",
    "reason",
]

df_missing_pages = pd.DataFrame(
    missing_page_rows,
    columns=report_columns,
)

df_missing_pages.to_csv(PAGE_REPAIR_REPORT, index=False)


# ------------------------------------------------------------
# Calculate issue-level statistics
# ------------------------------------------------------------

expected_issue_ids = set(df_pages["anno_id"])

existing_issue_dirs = {
    path.name
    for path in OUT_PAGES.iterdir()
    if path.is_dir()
}

completely_missing_issue_ids = sorted(
    expected_issue_ids - existing_issue_dirs
)

affected_issue_ids = sorted(
    df_missing_pages["anno_id"].dropna().unique()
) if not df_missing_pages.empty else []


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FAST LOCAL COMPLETENESS CHECK")
print("=" * 70)

print("Expected issues:", len(expected_issue_ids))
print("Existing issue folders:", len(existing_issue_dirs))
print(
    "Completely missing issue folders:",
    len(completely_missing_issue_ids),
)

print("\nExpected pages:", expected_pages)
print("Valid local pages:", valid_pages)
print("Missing page files:", missing_files)
print("Files below minimum size:", small_files)
print("Unreadable filesystem entries:", unreadable_files)

print(
    "Issues containing at least one problematic page:",
    len(affected_issue_ids),
)

print("\nRepair report written to:")
print(PAGE_REPAIR_REPORT)

if completely_missing_issue_ids:
    print("\nCompletely missing issues:")
    for anno_id in completely_missing_issue_ids[:20]:
        print(" ", anno_id)

    if len(completely_missing_issue_ids) > 20:
        print(
            f"  ... and "
            f"{len(completely_missing_issue_ids) - 20} more"
        )

print("=" * 70)

Indexing locally downloaded JPG files...


Indexing JPG files: 21303it [00:10, 2045.97it/s]
Checking expected pages: 100%|██████████| 21349/21349 [00:00<00:00, 57607.74it/s]



FAST LOCAL COMPLETENESS CHECK
Expected issues: 2593
Existing issue folders: 2593
Completely missing issue folders: 0

Expected pages: 21349
Valid local pages: 21303
Missing page files: 46
Files below minimum size: 0
Unreadable filesystem entries: 0
Issues containing at least one problematic page: 31

Repair report written to:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/missing_pages_to_redownload.csv


In [ ]:
# ============================================================
# REDOWNLOAD MISSING PAGES AND REPAIR PAGE METADATA
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MISSING_PAGES_CSV = (
    OUT_META / "missing_pages_to_redownload.csv"
)

REPAIR_LOG_CSV = (
    OUT_META / "missing_pages_repair_log.csv"
)

REQUEST_TIMEOUT = 120
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 3
MIN_FILE_SIZE_BYTES = 1024
CHUNK_SIZE = 1024 * 1024


# ------------------------------------------------------------
# Validate input files
# ------------------------------------------------------------

if not MISSING_PAGES_CSV.exists():
    raise FileNotFoundError(
        f"Missing repair report:\n{MISSING_PAGES_CSV}"
    )

if not PAGES_CSV.exists():
    raise FileNotFoundError(
        f"Missing page metadata CSV:\n{PAGES_CSV}"
    )


# ------------------------------------------------------------
# Load missing-page report
# ------------------------------------------------------------

df_missing = pd.read_csv(
    MISSING_PAGES_CSV,
    dtype=str
).fillna("")

required_report_columns = {
    "anno_id",
    "page_id",
    "iiif_jpg_url",
}

missing_report_columns = (
    required_report_columns - set(df_missing.columns)
)

if missing_report_columns:
    raise ValueError(
        "The missing-page report lacks these columns: "
        + ", ".join(sorted(missing_report_columns))
    )

df_missing["anno_id"] = (
    df_missing["anno_id"]
    .astype(str)
    .str.strip()
)

df_missing["page_id"] = (
    df_missing["page_id"]
    .astype(str)
    .str.strip()
)

df_missing["iiif_jpg_url"] = (
    df_missing["iiif_jpg_url"]
    .astype(str)
    .str.strip()
)

# Ignore malformed blank rows and accidental duplicate page IDs.
df_missing = (
    df_missing[
        df_missing["anno_id"].ne("")
        & df_missing["page_id"].ne("")
    ]
    .drop_duplicates(
        subset=["page_id"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("Pages listed in repair report:", len(df_missing))


# ------------------------------------------------------------
# Load page metadata
# ------------------------------------------------------------

df_pages = pd.read_csv(
    PAGES_CSV,
    dtype=str
).fillna("")

required_metadata_columns = {
    "anno_id",
    "page_id",
}

missing_metadata_columns = (
    required_metadata_columns - set(df_pages.columns)
)

if missing_metadata_columns:
    raise ValueError(
        "The page metadata CSV lacks these columns: "
        + ", ".join(sorted(missing_metadata_columns))
    )

df_pages["anno_id"] = (
    df_pages["anno_id"]
    .astype(str)
    .str.strip()
)

df_pages["page_id"] = (
    df_pages["page_id"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Ensure repair-related metadata columns exist
# ------------------------------------------------------------

for column in [
    "local_jpg_path",
    "download_status",
    "download_error",
    "repaired_at",
]:
    if column not in df_pages.columns:
        df_pages[column] = ""


# ------------------------------------------------------------
# Back up metadata before changing it
# ------------------------------------------------------------

backup_timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

metadata_backup = PAGES_CSV.with_name(
    f"{PAGES_CSV.stem}_backup_before_page_repair_"
    f"{backup_timestamp}.csv"
)

shutil.copy2(
    PAGES_CSV,
    metadata_backup
)

print("Metadata backup:")
print(metadata_backup)


# ------------------------------------------------------------
# Image-validation function
# ------------------------------------------------------------

def is_valid_jpeg(
    path: Path,
    minimum_size: int = MIN_FILE_SIZE_BYTES,
) -> tuple[bool, str]:
    """
    Check that a file exists, has a plausible size, and can be
    decoded as an image by Pillow.
    """

    if not path.exists():
        return False, "file does not exist"

    try:
        file_size = path.stat().st_size
    except OSError as exc:
        return False, f"cannot read file size: {exc}"

    if file_size < minimum_size:
        return False, (
            f"file is too small: {file_size} bytes"
        )

    try:
        with Image.open(path) as image:
            image.verify()

        # Reopen after verify() because verify invalidates the
        # first Pillow image object.
        with Image.open(path) as image:
            image.load()

            detected_format = (
                image.format or ""
            ).upper()

            if detected_format not in {
                "JPEG",
                "JPG",
            }:
                return False, (
                    "downloaded file is not a JPEG: "
                    f"{detected_format or 'unknown format'}"
                )

    except Exception as exc:
        return False, (
            f"image verification failed: "
            f"{type(exc).__name__}: {exc}"
        )

    return True, ""


# ------------------------------------------------------------
# Safe download function
# ------------------------------------------------------------

def download_jpeg(
    url: str,
    destination: Path,
) -> tuple[int, int]:
    """
    Download one JPEG safely.

    The response is first written to a temporary .part file.
    The final JPG is created only after the file passes image
    verification.

    Returns:
        downloaded file size,
        number of attempts used
    """

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temporary_path = destination.with_suffix(
        destination.suffix + ".part"
    )

    last_error = None

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):
        if temporary_path.exists():
            temporary_path.unlink()

        try:
            with requests.get(
                url,
                stream=True,
                timeout=REQUEST_TIMEOUT,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "IIIF-missing-page-repair/1.0"
                    )
                },
            ) as response:

                response.raise_for_status()

                content_type = (
                    response.headers
                    .get("Content-Type", "")
                    .lower()
                )

                if (
                    content_type
                    and "image" not in content_type
                    and "octet-stream" not in content_type
                ):
                    raise ValueError(
                        "Unexpected Content-Type: "
                        f"{content_type}"
                    )

                with temporary_path.open("wb") as file:
                    for chunk in response.iter_content(
                        chunk_size=CHUNK_SIZE
                    ):
                        if chunk:
                            file.write(chunk)

            valid, validation_error = (
                is_valid_jpeg(temporary_path)
            )

            if not valid:
                raise ValueError(validation_error)

            file_size = (
                temporary_path.stat().st_size
            )

            # Atomically replace an invalid or partial old file.
            temporary_path.replace(destination)

            return file_size, attempt

        except Exception as exc:
            last_error = exc

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt < MAX_RETRIES:
                time.sleep(
                    RETRY_DELAY_SECONDS * attempt
                )

    raise RuntimeError(
        f"Download failed after {MAX_RETRIES} attempts: "
        f"{type(last_error).__name__}: {last_error}"
    )


# ------------------------------------------------------------
# Download and repair
# ------------------------------------------------------------

repair_log = []
successful_page_ids = set()

for _, row in tqdm(
    df_missing.iterrows(),
    total=len(df_missing),
    desc="Repairing missing pages",
):
    anno_id = row["anno_id"]
    page_id = row["page_id"]
    jpg_url = row["iiif_jpg_url"]

    destination = (
        OUT_PAGES
        / anno_id
        / f"{page_id}.jpg"
    )

    log_entry = {
        "anno_id": anno_id,
        "page_id": page_id,
        "iiif_jpg_url": jpg_url,
        "local_jpg_path": str(destination),
        "status": "",
        "file_size_bytes": "",
        "attempts": "",
        "error": "",
    }

    # --------------------------------------------------------
    # If a valid file is already present, preserve it.
    # --------------------------------------------------------

    valid_existing, existing_error = (
        is_valid_jpeg(destination)
    )

    if valid_existing:
        log_entry["status"] = "already-valid"
        log_entry["file_size_bytes"] = (
            destination.stat().st_size
        )
        log_entry["attempts"] = 0

        successful_page_ids.add(page_id)
        repair_log.append(log_entry)
        continue

    # --------------------------------------------------------
    # The page cannot be downloaded without a URL.
    # --------------------------------------------------------

    if not jpg_url:
        log_entry["status"] = "failed"
        log_entry["error"] = (
            "The repair report contains no IIIF JPG URL."
        )

        repair_log.append(log_entry)
        continue

    # --------------------------------------------------------
    # Download or replace the missing/invalid file.
    # --------------------------------------------------------

    try:
        file_size, attempts_used = download_jpeg(
            jpg_url,
            destination,
        )

        log_entry["status"] = "downloaded"
        log_entry["file_size_bytes"] = file_size
        log_entry["attempts"] = attempts_used

        successful_page_ids.add(page_id)

    except Exception as exc:
        log_entry["status"] = "failed"
        log_entry["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    repair_log.append(log_entry)


# ------------------------------------------------------------
# Repair metadata for successfully restored pages
# ------------------------------------------------------------

repair_timestamp = datetime.now().isoformat(
    timespec="seconds"
)

metadata_rows_missing = []

for page_id in successful_page_ids:
    report_match = df_missing[
        df_missing["page_id"] == page_id
    ]

    if report_match.empty:
        continue

    report_row = report_match.iloc[0]

    anno_id = report_row["anno_id"]

    destination = (
        OUT_PAGES
        / anno_id
        / f"{page_id}.jpg"
    )

    metadata_mask = (
        df_pages["page_id"] == page_id
    )

    if not metadata_mask.any():
        metadata_rows_missing.append(page_id)
        continue

    # Update only this page's metadata row.
    df_pages.loc[
        metadata_mask,
        "local_jpg_path"
    ] = str(destination)

    df_pages.loc[
        metadata_mask,
        "download_status"
    ] = "ok"

    df_pages.loc[
        metadata_mask,
        "download_error"
    ] = ""

    df_pages.loc[
        metadata_mask,
        "repaired_at"
    ] = repair_timestamp

    # Restore the URL if the metadata row unexpectedly lacks it.
    if "iiif_jpg_url" in df_pages.columns:
        empty_url_mask = (
            metadata_mask
            & df_pages["iiif_jpg_url"]
                .astype(str)
                .str.strip()
                .eq("")
        )

        df_pages.loc[
            empty_url_mask,
            "iiif_jpg_url"
        ] = report_row["iiif_jpg_url"]


# ------------------------------------------------------------
# Mark failed pages in metadata
# ------------------------------------------------------------

failed_log_entries = [
    entry
    for entry in repair_log
    if entry["status"] == "failed"
]

for entry in failed_log_entries:
    metadata_mask = (
        df_pages["page_id"]
        == entry["page_id"]
    )

    if metadata_mask.any():
        df_pages.loc[
            metadata_mask,
            "download_status"
        ] = "repair-failed"

        df_pages.loc[
            metadata_mask,
            "download_error"
        ] = entry["error"]


# ------------------------------------------------------------
# Remove accidental duplicate metadata rows
# ------------------------------------------------------------

duplicate_count = int(
    df_pages.duplicated(
        subset=["page_id"],
        keep=False
    ).sum()
)

if duplicate_count:
    print(
        "\nWarning:",
        duplicate_count,
        "metadata rows had duplicate page IDs."
    )

    df_pages = (
        df_pages
        .drop_duplicates(
            subset=["page_id"],
            keep="last"
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Save repaired metadata and repair log
# ------------------------------------------------------------

df_pages.to_csv(
    PAGES_CSV,
    index=False
)

df_repair_log = pd.DataFrame(
    repair_log
)

df_repair_log.to_csv(
    REPAIR_LOG_CSV,
    index=False
)


# ------------------------------------------------------------
# Final local verification
# ------------------------------------------------------------

still_missing = []
still_invalid = []

for _, row in df_missing.iterrows():
    destination = (
        OUT_PAGES
        / row["anno_id"]
        / f"{row['page_id']}.jpg"
    )

    valid, validation_error = (
        is_valid_jpeg(destination)
    )

    if not destination.exists():
        still_missing.append(
            row["page_id"]
        )

    elif not valid:
        still_invalid.append({
            "page_id": row["page_id"],
            "error": validation_error,
        })


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

downloaded_count = sum(
    entry["status"] == "downloaded"
    for entry in repair_log
)

already_valid_count = sum(
    entry["status"] == "already-valid"
    for entry in repair_log
)

failed_count = sum(
    entry["status"] == "failed"
    for entry in repair_log
)

print("\n" + "=" * 70)
print("MISSING-PAGE REPAIR RESULTS")
print("=" * 70)

print("Pages in repair report:", len(df_missing))
print("Pages downloaded:", downloaded_count)
print("Pages already valid:", already_valid_count)
print("Failed downloads:", failed_count)

print(
    "Pages still absent after repair:",
    len(still_missing)
)

print(
    "Pages still invalid after repair:",
    len(still_invalid)
)

print("\nRepaired metadata:")
print(PAGES_CSV)

print("\nRepair log:")
print(REPAIR_LOG_CSV)

print("\nMetadata backup:")
print(metadata_backup)

if metadata_rows_missing:
    print(
        "\nWarning: downloaded pages without matching "
        "metadata rows:"
    )

    for page_id in metadata_rows_missing:
        print(" ", page_id)

if failed_log_entries:
    print("\nFailed downloads:")

    for entry in failed_log_entries:
        print(
            f"  {entry['page_id']}: "
            f"{entry['error']}"
        )

print("=" * 70)

Pages listed in repair report: 46
Metadata backup:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata_backup_before_page_repair_20260726_093551.csv


Repairing missing pages: 100%|██████████| 46/46 [02:29<00:00,  3.26s/it]



MISSING-PAGE REPAIR RESULTS
Pages in repair report: 46
Pages downloaded: 46
Pages already valid: 0
Failed downloads: 0
Pages still absent after repair: 0
Pages still invalid after repair: 0

Repaired metadata:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv

Repair log:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/missing_pages_repair_log.csv

Metadata backup:
/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata_backup_before_page_repair_20260726_093551.csv


In [ ]:
# ======================
# Final Integrity Check
# ======================

METADATA_CSV = Path(
    "/content/drive/MyDrive/Masterarbeit_DH/Pipeline_building-character/Data/Data_acquisition_BOMBE/metadata/pages-jpg_metadata.csv"
)

# ========= Load Metadata =========
df = pd.read_csv(METADATA_CSV, dtype=str).fillna("")

# ========= File Counts =========
img_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

all_imgs = [
    p for p in OUT_PAGES.rglob("*")
    if p.is_file() and p.suffix.lower() in img_exts
]

issue_dirs = [
    p for p in OUT_PAGES.iterdir()
    if p.is_dir()
]

# ========= Counts =========
metadata_rows = len(df)

unique_page_ids = (
    df["page_id"].astype(str).str.strip().nunique()
)

duplicate_page_ids = metadata_rows - unique_page_ids

image_files = len(all_imgs)

unique_stems = len({p.stem for p in all_imgs})
duplicate_stems = image_files - unique_stems

# ========= Download Status =========
status_counts = (
    df["download_status"]
      .astype(str)
      .str.strip()
      .value_counts(dropna=False)
)

ok_count = (
    df["download_status"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("ok")
      .sum()
)

# ========= CSV ↔ File Consistency =========
csv_page_ids = set(
    df["page_id"]
      .astype(str)
      .str.strip()
)

file_stems = {
    p.stem
    for p in all_imgs
}

metadata_without_file = csv_page_ids - file_stems
files_without_metadata = file_stems - csv_page_ids

# ========= Report =========
print("=" * 70)
print("FINAL DATA ACQUISITION VALIDATION")
print("=" * 70)

print("\n--- ISSUE FOLDERS ---")
print("Issue folders:", len(issue_dirs))

print("\n--- METADATA ---")
print("Rows:", metadata_rows)
print("Unique page_ids:", unique_page_ids)
print("Duplicate page_ids:", duplicate_page_ids)

print("\n--- IMAGE FILES ---")
print("Image files:", image_files)
print("Unique stems:", unique_stems)
print("Duplicate stems:", duplicate_stems)

print("\n--- DOWNLOAD STATUS ---")
print(status_counts)

print("\n--- CSV ↔ FILE CONSISTENCY ---")
print("Metadata page_ids without image:", len(metadata_without_file))
print("Image files without metadata row:", len(files_without_metadata))

# ========= Show potential problems =========
if metadata_without_file:
    print("\nFirst metadata rows without image:")
    for x in sorted(list(metadata_without_file))[:20]:
        print("  ", x)

if files_without_metadata:
    print("\nFirst image files without metadata:")
    for x in sorted(list(files_without_metadata))[:20]:
        print("  ", x)

# ========= Summary =========
print("\n" + "=" * 70)

passed = (
    duplicate_page_ids == 0
    and duplicate_stems == 0
    and len(metadata_without_file) == 0
    and len(files_without_metadata) == 0
)

if passed:
    print("VALIDATION PASSED")
    print("Metadata and image files are fully consistent.")
else:
    print("VALIDATION FOUND ISSUES")
    print("Inspect the counts above.")

print("=" * 70)

FINAL DATA ACQUISITION VALIDATION

--- ISSUE FOLDERS ---
Issue folders: 2593

--- METADATA ---
Rows: 21349
Unique page_ids: 21349
Duplicate page_ids: 0

--- IMAGE FILES ---
Image files: 21349
Unique stems: 21349
Duplicate stems: 0

--- DOWNLOAD STATUS ---
download_status
ok    21349
Name: count, dtype: int64

--- CSV ↔ FILE CONSISTENCY ---
Metadata page_ids without image: 0
Image files without metadata row: 0

VALIDATION PASSED
Metadata and image files are fully consistent.
